# 실습 A: OpenAlex 연도별 건수와 천 건당 비율

『대중천문학개론』 10장 6.1절 실습 A의 코드 전체. `OFFLINE = True`이면 `data/fig10-3.csv`(2026-10-14 조회값)로, `False`이면 OpenAlex API에서 새로 받는다. 무료 키가 있으면 환경 변수 `OPENALEX_API_KEY`로 넘긴다.

In [ ]:
OFFLINE = True   # 인터넷이 되면 False로 바꾼다

In [ ]:
# Colab에서 열었으면 저장소를 받아 ch10 폴더로 옮긴다(로컬에서는 아무 일도 하지 않는다)
import os, subprocess, sys
if "google.colab" in sys.modules and not os.path.exists("README.md"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/Park-Jiwon-Maker/public-astronomy-textbook.git", "repo"], check=True)
    os.chdir("repo/ch10")
print(os.getcwd())

In [ ]:
if OFFLINE:                      # 인터넷 대신 data/ 폴더의 고정 자료로 응답하게 한다
    import offline_fixtures
    offline_fixtures.install()

In [ ]:
# 실습 A: OpenAlex 연도별 건수와 천 건당 비율
import os
import requests
import pandas as pd
from urllib.parse import quote

URL = "https://api.openalex.org/works"
KEY = os.environ.get("OPENALEX_API_KEY")   # 무료 키(선택). 키를 코드에 직접 적지 않는다

def by_year(query):
    f = quote(f"title_and_abstract.search:{query},publication_year:2000-2025", safe=":,")
    r = requests.get(f"{URL}?filter={f}&group_by=publication_year",
                     params={"api_key": KEY} if KEY else {}, timeout=30)
    r.raise_for_status()
    js = r.json()
    print(query, js["meta"]["count"], js["meta"].get("cost_usd"))   # 질의 기록표로 옮긴다
    return pd.Series({int(g["key"]): g["count"] for g in js["group_by"]}).sort_index()

num = by_year('"astronomy education"')
den = by_year("astronomy")
num = num.reindex(den.index, fill_value=0)   # 0건인 해는 group_by 결과에 빠지므로 0으로 채운다
print((num / den * 1000).round(2).loc[2020:])   # astronomy 문헌 천 건당 비율